In [1]:
import ROOT
import numpy as np
import math


def root_minimize(func,
                ndim,
                minimizerName="Minuit2",
                algoName="",
                mg_init=None,
                eps_init=None,
                a1_init=None,
                a2_init=None,
                stepSize=None,
                maxFunctionCalls=1000000,
                maxIterations=10000,
                tolerance=1e-8,
                printLevel=1):
    """
    Generic ROOT Minimization Wrapper with Confidence Level Calculation.

    Parameters
    ----------
    func : callable
        Function to minimize. Should accept a list or numpy array of length `ndim`.

    ndim : int
        Number of parameters to minimize.

    minimizerName : str, default="Minuit2"
        Minimizer to use (Minuit, Minuit2, GSLMultiMin, GSLSimAn, Genetic, etc.).
    
    algoName : str, default=""
        Specific algorithm (Migrad, BFGS, ConjugateFR, Simplex, etc.).
    
    mg_init : float or None
        Initial guess for parameter 'mg'. Defaults to 0.0 if None.
    
    eps_init : float or None
        Initial guess for parameter 'eps'. Defaults to 0.0 if None.
    
    a1_init : float or None
        Initial guess for parameter 'a1'. Defaults to 0.0 if None.
    
    a2_init : float or None
        Initial guess for parameter 'a2'. Defaults to 0.0 if None.
    
    stepSize : list of floats or None
        Step sizes for each parameter. Defaults to 0.01 for all.
    
    maxFunctionCalls : int, default=1000000
        Maximum allowed function evaluations.
    
    maxIterations : int, default=10000
        Maximum allowed iterations.
    
    tolerance : float, default=1e-8
        Desired tolerance for convergence.
    
    printLevel : int, default=1
        Verbosity of the minimizer (0=quiet, 1=normal, 2=verbose).

    Returns
    -------
    dict
        Dictionary containing:
        - 'success': bool, whether minimization converged successfully
        - 'x': numpy array, parameter values at minimum
        - 'status': int, minimizer status (0 = success)
        - 'hesse_errors': numpy array, symmetric Hesse errors
        - 'minos_errors_low': numpy array, lower MINOS errors
        - 'minos_errors_up': numpy array, upper MINOS errors
    """

    #-------------------
    #  SET STARTING POINT
    #-------------------

    param_names = ["mg", "eps", "a1", "a2"]
    init_map = [mg_init, eps_init, a1_init, a2_init]
    startPoint = []
    for i in range(ndim):
        if init_map[i] is not None:
            startPoint.append(init_map[i])
        else:
            startPoint.append(0.0)  # fallback default
    # --------------------------------------------------------------

    #-------------------
    #  SET STEP SIZE
    #-------------------
    if stepSize is None:
        stepSize = [0.01] * ndim

    #-------------------
    #  SET CONFIDENCE LEVEL FOR 4D CASE 90% CL
    #-------------------
    errordef = 7.78

    # import scipy
    # import scipy.stats
    # print(scipy.stats.chi2.ppf(0.9 , df=4))

    # cl_to_errordef_4d = {
    #     68.3: 4.72,
    #     90.0: 7.78,
    #     95.0: 9.49,
    #     99.0: 13.28
    # }

    
    #-------------------
    #  CREATE MINIMIZER
    #-------------------

    minimizer = ROOT.Math.Factory.CreateMinimizer(minimizerName, algoName)
    if not minimizer:
        raise RuntimeError(f"Cannot create minimizer \"{minimizerName}\"")
    

    #-------------------
    #  SET OPTIONS
    #-------------------

    minimizer.SetMaxFunctionCalls(maxFunctionCalls)
    minimizer.SetMaxIterations(maxIterations)
    minimizer.SetTolerance(tolerance)
    minimizer.SetPrintLevel(printLevel)
    minimizer.SetErrorDef(errordef)
    f = ROOT.Math.Functor(func, ndim)
    minimizer.SetFunction(f)

    
    variable = list(startPoint)

    #-------------------
    #  SET PARAMETERS
    #-------------------

    # renaming variable names to match model parameters and set parameters 
    for i in range(ndim):
        if i < len(param_names):
            name = param_names[i]
        else:
            name = f"x{i}"
        minimizer.SetVariable(i, name, variable[i], stepSize[i])

    #could replace the code above by the following code to set parameters without renaming

    # for i in range(ndim):
    #     minimizer.SetVariable(i, f"x{i}", variable[i], stepSize[i])


    #-------------------
    #  RUN MINIMIZATION
    #-------------------

    minimization = minimizer.Minimize()
    if not minimization:
        return {'success': False}
    

    #-------------------
    # GET HESSE ERROR
    #-------------------

    # Create empty arrays to store the results
    xs = np.zeros(ndim)           # parameter values at minimum
    hesse_errors = np.zeros(ndim) # symmetric Hesse errors

    # Loop over each parameter and extract the value and Hesse error
    for i in range(ndim):
        xs[i] = minimizer.X()[i]          # get the fitted value of parameter i
        hesse_errors[i] = minimizer.Errors()[i]  # get the Hesse error for parameter i


    #-------------------
    # GET MINOS ERROR
    #-------------------
    
    # Initialize arrays to store MINOS errors
    minos_errors_low = np.zeros(ndim)
    minos_errors_up = np.zeros(ndim)

    # Temporary arrays for ROOT's GetMinosError
    errLow = np.zeros(1, dtype=np.float64)
    errUp  = np.zeros(1, dtype=np.float64)

    for i in range(ndim):
        success = minimizer.GetMinosError(i, errLow, errUp)
        if success:
            minos_errors_low[i] = errLow[0]
            minos_errors_up[i] = errUp[0]
        else:
            # fallback to Hesse errors if MINOS fails
            minos_errors_low[i] = -hesse_errors[i]
            minos_errors_up[i] = hesse_errors[i]



    # print results
    print("\nMinimization results (values ± Hesse ± MINOS):")
    for i in range(ndim):
        print(f"{param_names[i]}: {xs[i]:.6f} "
              f"± {hesse_errors[i]:.6f} "
              f"[{minos_errors_low[i]:+.6f}, {minos_errors_up[i]:+.6f}]")

    print(f"\nStatus: {minimizer.Status()} (0 = success)\n")
    # ----------------------

    return {
        'success': minimization and minimizer.Status() == 0,
        'x': xs,
        'status': minimizer.Status(),
        'hesse_errors': hesse_errors,
        'minos_errors_low': minos_errors_low,
        'minos_errors_up': minos_errors_up,
    }


In [2]:
import ROOT
import numpy as np

def model_function(x, par):
    """
    Função modelo COMPLEXA com 4 parâmetros livres:
    f(x) = mg * exp(-eps * x[0]) + a1 * x[0] + a2 * x[0]**2
    
    par[0] = mg  (magnitude/amplitude)
    par[1] = eps (taxa de decaimento exponencial)
    par[2] = a1  (coeficiente linear)
    par[3] = a2  (coeficiente quadrático)
    """
    mg = par[0]
    eps = par[1]
    a1 = par[2]
    a2 = par[3]

    # Prevenir overflow no exp
    arg = eps * (x[0] - a1)
    if arg > 100:
        logistic = 0.0
    elif arg < -100:
        logistic = 1.0
    else:
        logistic = 1.0 / (1.0 + ROOT.TMath.Exp(arg))

    return mg * logistic + a2 * x[0]

def fit_data(func_model, x_data, y_data, y_errors, initial_params, param_limits, xmin, xmax):
    """
    Função para realizar o ajuste usando LeastSquareFit com ROOT
    """
    # Criar TGraphErrors com os dados
    n_points = len(x_data)
    graph = ROOT.TGraphErrors(n_points)
    
    for i in range(n_points):
        graph.SetPoint(i, x_data[i], y_data[i])
        graph.SetPointError(i, 0, y_errors[i])
    
    # Preparar o BinData
    data_range = ROOT.Fit.DataRange(xmin, xmax)
    data_options = ROOT.Fit.DataOptions()
    
    bin_data = ROOT.Fit.BinData(data_options, data_range)
    ROOT.Fit.FillData(bin_data, graph)
    
    # Criar a função TF1 com o modelo fornecido
    npar = len(initial_params)
    func = ROOT.TF1("fit_func", func_model, xmin, xmax, npar)
    
    # Nomear os parâmetros
    param_names = ['mg', 'eps', 'a1', 'a2']
    for i, name in enumerate(param_names):
        func.SetParName(i, name)
    
    # Definir parâmetros iniciais e limites
    for i, param in enumerate(initial_params):
        func.SetParameter(i, param)
        if param_limits[i] is not None:
            func.SetParLimits(i, param_limits[i][0], param_limits[i][1])
    
    # Criar wrapper e configurar o fitter
    wrapped_func = ROOT.Math.WrappedTF1(func)
    fitter = ROOT.Fit.Fitter()
    
    # Configurações do minimizador Minuit2
    fitter.Config().SetMinimizer("Minuit2", "Migrad")
    fitter.Config().MinimizerOptions().SetPrintLevel(0)
    fitter.Config().MinimizerOptions().SetStrategy(2)
    fitter.Config().MinimizerOptions().SetPrecision(1e-8)  
    fitter.Config().MinimizerOptions().SetTolerance(1e-3)
    fitter.Config().MinimizerOptions().SetMaxFunctionCalls(50000)
    fitter.Config().MinimizerOptions().SetMaxIterations(50000)
    
    fitter.SetFunction(wrapped_func, False)
    
    # Executar o Least Square Fit
    fit_result = fitter.LeastSquareFit(bin_data)
    
    # Extrair resultados
    result = fitter.Result()
    
    output = {
        'chi2': result.Chi2(),
        'ndf': result.Ndf(),
        'chi2_dof': result.Chi2() / result.Ndf() if result.Ndf() > 0 else 0.0,
        'parameters': [result.Parameter(i) for i in range(result.NPar())],
        'errors': [result.ParError(i) for i in range(result.NPar())],
        'param_names': param_names,
        'valid': result.IsValid(),
        'status': result.Status()
    }
    
    return output

# =============================================
# EXEMPLO COM FUNÇÃO COMPLEXA - 4 PARÂMETROS
# =============================================


true_params = {'mg': 8.0, 'eps': 1.2, 'a1': 5.0, 'a2': 0.3}
np.random.seed(42)
x_data = np.linspace(0.5, 10.0, 30)

# Dados simulados da função estável
y_true = np.array([model_function([x], list(true_params.values())) for x in x_data])

noise_level = 0.3
y_data = y_true + np.random.normal(0, noise_level, len(x_data))
y_errors = np.full(len(x_data), noise_level)

# Chutes iniciais razoáveis
initial_params = [5.0, 0.5, 4.0, 0.1]
param_limits = [
    (0.1, 50.0),     # mg
    (0.01, 10.0),    # eps
    (0.1, 10.0),     # a1
    (-2.0, 2.0)      # a2
]

# Executar o ajuste
result = fit_data(model_function, x_data, y_data, y_errors, initial_params, 
                  param_limits, x_data.min(), x_data.max())

# =============================================
# APRESENTAÇÃO SIMPLIFICADA DOS RESULTADOS
# =============================================

print("=" * 60)
print("RESULTADOS DO AJUSTE")
print("=" * 60)

if result['valid']:
    print(f"\nChi²/DOF = {result['chi2_dof']:.4f}\n")
    
    print("PARÂMETROS AJUSTADOS:")
    for i, name in enumerate(result['param_names']):
        param_val = result['parameters'][i]
        param_err = result['errors'][i]
        print(f"  {name:4s} = {param_val:8.4f} ± {param_err:.4f}")
else:
    print(f"❌ AJUSTE FALHOU! (Status: {result['status']})")
    print("\nParâmetros tentados:")
    for i, name in enumerate(result['param_names']):
        print(f"  {name:4s} = {result['parameters'][i]:8.4f}")
    
print("=" * 60)

RESULTADOS DO AJUSTE

Chi²/DOF = 0.6305

PARÂMETROS AJUSTADOS:
  mg   =   8.2104 ± 0.1410
  eps  =   1.1953 ± 0.0813
  a1   =   4.8639 ± 0.0665
  a2   =   0.2917 ± 0.0127


In [3]:
import ROOT
import numpy as np
import math

def model_function(x, par):
    """
    Função modelo COMPLEXA com 4 parâmetros livres:
    f(x) = mg * exp(-eps * x[0]) + a1 * x[0] + a2 * x[0]**2
    
    par[0] = mg  (magnitude/amplitude)
    par[1] = eps (taxa de decaimento exponencial)
    par[2] = a1  (coeficiente linear)
    par[3] = a2  (coeficiente quadrático)
    """
    mg = par[0]
    eps = par[1]
    a1 = par[2]
    a2 = par[3]

    # Prevenir overflow no exp
    arg = eps * (x[0] - a1)
    if arg > 100:
        logistic = 0.0
    elif arg < -100:
        logistic = 1.0
    else:
        logistic = 1.0 / (1.0 + ROOT.TMath.Exp(arg))

    return mg * logistic + a2 * x[0]


def fit_data_with_minimizer(x_data, y_data, y_errors, initial_params, param_limits=None):
    """
    Função para realizar o ajuste usando LeastSquareFit e root_minimize (Code 2)
    """
    # Criar TGraphErrors com os dados
    n_points = len(x_data)
    graph = ROOT.TGraphErrors(n_points)
    
    for i in range(n_points):
        graph.SetPoint(i, x_data[i], y_data[i])
        graph.SetPointError(i, 0, y_errors[i])
    
    # Preparar o BinData
    xmin, xmax = x_data.min(), x_data.max()
    data_range = ROOT.Fit.DataRange(xmin, xmax)
    data_options = ROOT.Fit.DataOptions()
    
    bin_data = ROOT.Fit.BinData(data_options, data_range)
    ROOT.Fit.FillData(bin_data, graph)
    
    # Criar a função TF1 com o modelo fornecido
    npar = len(initial_params)
    func = ROOT.TF1("fit_func", model_function, xmin, xmax, npar)
    
    # Nomear os parâmetros
    param_names = ['mg', 'eps', 'a1', 'a2']
    for i, name in enumerate(param_names):
        func.SetParName(i, name)
    
    # Definir parâmetros iniciais e limites
    for i, param in enumerate(initial_params):
        func.SetParameter(i, param)
        if param_limits is not None and param_limits[i] is not None:
            func.SetParLimits(i, param_limits[i][0], param_limits[i][1])
    
    # Criar wrapper e configurar o fitter
    wrapped_func = ROOT.Math.WrappedTF1(func)
    fitter = ROOT.Fit.Fitter()
    
    # Configurações do minimizador Minuit2
    fitter.Config().SetMinimizer("Minuit2", "Migrad")
    fitter.Config().MinimizerOptions().SetPrintLevel(1)
    fitter.Config().MinimizerOptions().SetStrategy(2)
    fitter.Config().MinimizerOptions().SetPrecision(1e-8)  
    fitter.Config().MinimizerOptions().SetTolerance(1e-3)
    fitter.Config().MinimizerOptions().SetMaxFunctionCalls(50000)
    fitter.Config().MinimizerOptions().SetMaxIterations(50000)
    
    fitter.SetFunction(wrapped_func, False)
    
    # Executar o Least Square Fit
    fit_result = fitter.LeastSquareFit(bin_data)
    
    # Extrair resultados básicos
    result = fitter.Result()
    
    # Extract initial parameters for root_minimize
    mg_init = result.Parameter(0)
    eps_init = result.Parameter(1)
    a1_init = result.Parameter(2)
    a2_init = result.Parameter(3)
    
    # Chi-square function for root_minimize
    def chi2(params):
        chi2_sum = 0.0
        for i in range(len(x_data)):
            y_pred = model_function([x_data[i]], params)
            residual = (y_data[i] - y_pred) / y_errors[i]
            chi2_sum += residual * residual
        return chi2_sum
    
    # Call root_minimize for MINOS errors
    minos_result = root_minimize(
        func=chi2,
        ndim=4,
        minimizerName="Minuit2",
        algoName="Migrad",
        mg_init=mg_init,
        eps_init=eps_init,
        a1_init=a1_init,
        a2_init=a2_init,
        stepSize=[0.1, 0.01, 0.1, 0.01],
        maxFunctionCalls=50000,
        maxIterations=50000,
        tolerance=1e-8,
        printLevel=1
    )
    
    return {
        'chi2': result.Chi2(),
        'ndf': result.Ndf(),
        'chi2_dof': result.Chi2() / result.Ndf() if result.Ndf() > 0 else 0.0,
        'parameters': [result.Parameter(i) for i in range(result.NPar())],
        'errors': [result.ParError(i) for i in range(result.NPar())],
        'minos_errors_low': minos_result['minos_errors_low'],
        'minos_errors_up': minos_result['minos_errors_up'],
        'param_names': param_names,
        'valid': result.IsValid() and minos_result['success'],
        'status': result.Status()
    }


# =============================================
# EXEMPLO COM FUNÇÃO COMPLEXA - 4 PARÂMETROS
# =============================================

true_params = {'mg': 8.0, 'eps': 1.2, 'a1': 5.0, 'a2': 0.3}
np.random.seed(42)
x_data = np.linspace(0.5, 10.0, 30)

# Dados simulados da função estável
y_true = np.array([model_function([x], list(true_params.values())) for x in x_data])

noise_level = 0.3
y_data = y_true + np.random.normal(0, noise_level, len(x_data))
y_errors = np.full(len(x_data), noise_level)

# Chutes iniciais razoáveis
initial_params = [5.0, 0.5, 4.0, 0.1]
param_limits = [
    (0.1, 50.0),     # mg
    (0.01, 10.0),    # eps
    (0.1, 10.0),     # a1
    (-2.0, 2.0)      # a2
]

# Executar o ajuste usando o minimizador do Code 2
result = fit_data_with_minimizer(x_data, y_data, y_errors, initial_params)

# =============================================
# APRESENTAÇÃO SIMPLIFICADA DOS RESULTADOS
# =============================================

print("=" * 60)
print("RESULTADOS DO AJUSTE (usando root_minimize)")
print("=" * 60)

if result['valid']:
    print(f"\nChi²/DOF = {result['chi2_dof']:.4f}")
    print(f"Chi² = {result['chi2']:.4f}")
    print(f"NDF = {result['ndf']}\n")
    
    print("PARÂMETROS AJUSTADOS (com erros HESSE e MINOS):")
    for i, name in enumerate(result['param_names']):
        param_val = result['parameters'][i]
        hesse_err = result['errors'][i]
        minos_low = result['minos_errors_low'][i]
        minos_up = result['minos_errors_up'][i]
        print(f"  {name:4s} = {param_val:8.4f} ± {hesse_err:.4f} [{minos_low:+.4f}, {minos_up:+.4f}]")
else:
    print(f"❌ AJUSTE FALHOU! (Status: {result['status']})")
    print("\nParâmetros tentados:")
    for i, name in enumerate(result['param_names']):
        print(f"  {name:4s} = {result['parameters'][i]:8.4f}")
    
print("=" * 60)


Minimization results (values ± Hesse ± MINOS):
mg: 8.210428 ± 0.394073 [-0.374398, +0.422138]
eps: 1.195198 ± 0.227503 [-0.208032, +0.253587]
a1: 4.863867 ± 0.185075 [-0.186864, +0.185785]
a2: 0.291741 ± 0.035384 [-0.036525, +0.034555]

Status: 0 (0 = success)

RESULTADOS DO AJUSTE (usando root_minimize)

Chi²/DOF = 0.6305
Chi² = 16.3926
NDF = 26

PARÂMETROS AJUSTADOS (com erros HESSE e MINOS):
  mg   =   8.2104 ± 0.1410 [-0.3744, +0.4221]
  eps  =   1.1953 ± 0.0813 [-0.2080, +0.2536]
  a1   =   4.8639 ± 0.0665 [-0.1869, +0.1858]
  a2   =   0.2917 ± 0.0127 [-0.0365, +0.0346]
Minuit2Minimizer: Minimize with max-calls 50000 convergence for edm < 0.001 strategy 2
Minuit2Minimizer : Valid minimum - status = 0
FVAL  = 16.3925720298251676
Edm   = 4.25461452215469749e-07
Nfcn  = 204
Par_0	  = 8.21037	 +/-  0.141045
Par_1	  = 1.19525	 +/-  0.0812832
Par_2	  = 4.86387	 +/-  0.0664653
Par_3	  = 0.291745	 +/-  0.0126654
Minuit2Minimizer: Minimize with max-calls 50000 convergence for edm < 1e-08 

Warning in <Minuit2>: MnPosDef Matrix forced pos-def by adding to diagonal 0.708415
